In [1]:
import torch
import yaml
import pickle
from tqdm import tqdm
import pandas as pd
import soundfile as sf
import numpy as np
from models import BaselineModel, LinSeqModel, CNNModel, SklearnAudioClassifier
from baseline_experiment import BaselineExperiment as Exp

MODEL_REGISTRY = {
    'BaselineModel': BaselineModel,
    'LinSeqModel': LinSeqModel,
    'CNNModel': CNNModel,
}


In [2]:

# load config
CONFIG_PATH = 'config/pca_svm.yaml'  # or 'config/cnn.yaml' for PyTorch
with open(CONFIG_PATH, 'r') as file:
    config = yaml.safe_load(file)

model_name = config.get('model', 'BaselineModel')
is_sklearn = model_name == 'SklearnAudioClassifier'

if is_sklearn:
    # load sklearn model from pickle
    CKPT_PATH = '../ckpts/sklearn/pca_svm_20240101_120000-val_acc=0.70.ckpt'  # Update with actual path
    with open(CKPT_PATH, 'rb') as f:
        ckpt = pickle.load(f)

    model = SklearnAudioClassifier(
        classifier_type=ckpt['classifier_type'],
        sample_rate=ckpt['mel_transform_config']['sample_rate'],
        n_mels=ckpt['mel_transform_config']['n_mels'],
    )
    model.pipeline = ckpt['pipeline']
else:
    # load PyTorch model
    ModelClass = MODEL_REGISTRY[model_name]
    config['network']['sample_rate'] = config['data']['sample_rate']
    config['network']['n_label'] = config['experiment']['n_label']

    CKPT_PATH = f'../ckpts/{model_name.lower()}/epoch=4-val_acc=0.70.ckpt'
    model = ModelClass(**config['network'])
    exp = Exp.load_from_checkpoint(
        CKPT_PATH,
        model=model,
        **config['experiment'],
    )
    exp.eval()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/osperaja/PycharmProjects/dlap/dcase/src/../ckpts/baseline/epoch=4-val_acc=0.70.ckpt'

In [ ]:
TEST_PATH = f'./../../data/dlap/test1/'
meta_df = pd.read_csv(
    TEST_PATH + 'meta_blind.txt', delimiter='\t', header=None, names=['audio_path']
)
n_samples = len(meta_df)
class_labels = np.empty(n_samples, dtype=int)

for i in tqdm(range(n_samples)):
    audio_data, audio_sample_rate = sf.read(
        TEST_PATH + meta_df.iloc[i]['audio_path'], dtype=np.float32
    )
    if config['data']['mono']:
        audio_data = audio_data.mean(axis=-1, keepdims=True)
    audio_data = torch.from_numpy(audio_data.T)  # (CHANNEL, SAMPLES)

    if is_sklearn:
        features = model.extract_features(audio_data[None, :])
        est_label = model.pipeline.predict(features)[0]
    else:
        audio_data = audio_data.to(device=exp.device)
        with torch.no_grad():
            output = exp.model(audio_data[None, :])
            logits = output["logits"] if isinstance(output, dict) else output
        est_label = torch.argmax(logits, dim=-1).squeeze(dim=0)
        est_label = torch.bincount(est_label).argmax().cpu().item()

    class_labels[i] = est_label

# store results
meta_df['class_labels'] = class_labels
GROUPNAME = 'fibonacci'
meta_df.to_csv(
    TEST_PATH + f'{GROUPNAME}_test1.csv', sep='\t', header=None, index=False
)